# Session 3 — The LLM-based agent: retrieve, remember, plan, then act

**Salih Tutun, PhD** · DAT 5668 · Module 2 · Concept session (about 80 minutes)

I am Salih Tutun. Same Lumen Home desk as Monday. Maya's mug, Jordan's change of mind, Priya's order 1003. We are not leaving Agentic AI. Retrieval, memory, and a plan are how this **LLM-based agent** observes and decides. It still has to **act**.

Until I say **open the notebook**, this file stays closed. **Do not scroll.** One cell at a time. Guess to a neighbor (20 seconds). Two voices share. Then type.

**Family this session: LLM-based agent.** Session 4 is the other family (RL), still the same loop. Do not mix them because both are "AI."

![The agent chooses a; the environment returns r and s'](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/desk-agent-env.png)

![Same loop: RAG in observe, bandit in decide](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/desk-loop-where.png)

| Word | Meaning | Session 3 (today) | Session 4 (Nov 9) |
| --- | --- | --- | --- |
| **AGENT** | Chooses action *a* | Retrieve + plan + reviewer | ε-greedy (the **bandit**) |
| **ENVIRONMENT** | Returns *r* and next *s′* | CFPB queue + leakage | Frankfurter overnight return |
| State *s* | What the environment shows now | Maya's ticket, then a CFPB complaint | Today's visit: which FX sleeve |
| Observe | Tools *inside the agent* | **RAG** (split → index → retrieve → augment) and `lookup_order` | Only the sleeve you picked comes back |
| Decide | Policy *inside the agent* | Plan: cite or miss, then who releases | ε-greedy |
| Action *a* | What the agent sends into the environment | Refund, refuse, escalate | EUR, GBP, or JPY |
| Reward *r* | What the environment sends back | Was the citation real? Leakage? | That sleeve's overnight return |
| Next *s′* | What the environment shows next | The next ticket | Tomorrow's visit |

The agent is not the CSV. The environment is not RAG. RAG never picks the FX sleeve. A bandit never cites P1.

## Run of show
- 10 min — Recap Monday. Cold call: what did `lookup_order` observe that the chatbot skipped?
- 15 min — Why retrieval exists. Guess the SOC 2 question **before** we run it.
- 20 min — Near miss: which chunk comes first? Then the three-role handoff: who releases?
- 20 min — Your question. Then Chase 9999983 through s, a, r, s′. One framework line.
- 15 min — Walking-out answers, and why Session 4 is not this job.

We will not list the walking-out answers at the top. You will earn them.


## How we work in this notebook

I am Salih. I talk. You guess to a neighbor. Two voices share. Then we type **this cell only**. Do not scroll.

| You will see | What you do in class |
| --- | --- |
| **STOP** | Wait. Do not run the next cell until I say. |
| **This cell — ...** | I tell you what we are looking at. Then we run it together. |
| **YOU FILL THIS** | Guess to a neighbor (20 seconds). Two people share. Then type `YOU TYPE HERE`. Then run. |
| **RUN WITH ME** | Already written. Run it. Do not edit. |
| **Family check** | Say **LLM**, **RL**, or **hybrid** out loud — and one line you will keep for your project. |

There is no `TODO` in a concept session. Practice sessions (even numbers) are where you write the function.


## 1. Why retrieval exists — RUN WITH ME

A model that answers from memory will invent a refund window. Retrieval-augmented generation (RAG) is four steps, then the **agent still acts**.

![RAG is a tool inside observe](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/desk-rag-four.png)

1. **Split** the policy into chunks (P1–P4). One rule per chunk so we can cite it.
2. **Index** those chunks (today: a Python list; a vector store is the same box).
3. **Retrieve** top-*k* for *this* ticket.
4. **Augment + generate**: answer only from those pages. Then stamp action *a*. Reward *r* is whether the citation was real.

Today's ranker is **shared words**, on purpose, so a near miss is visible. A DAT 5565 embedding can replace the ranker — still a tool, not the agent. We are not installing LangChain today.

![Ask, cite, or miss](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/desk-policy-chunks.png)

Same instructional policy as Monday's desk (not a real firm). P1 is Maya's cracked mug. P2 is Jordan's change of mind.

Run this cell. Do not edit it. Then **STOP**.


In [ ]:
# RUN WITH ME. Do not edit this cell. This is RAG inside observe.
# Split: each dict is one policy chunk we can cite (P1–P4).
# Index: today that index is this Python list (a vector store is the same box).
CHUNKS = [
    {"id": "P1", "source": "Refund policy", "text": "A refund for damaged items is allowed within 30 days of delivery. The amount is the order total."},
    {"id": "P2", "source": "Refund policy", "text": "A refund for a change of mind is not allowed. Offer store credit within 14 days instead."},
    {"id": "P3", "source": "Privacy", "text": "Do not paste customer addresses or payment numbers into a public chatbot."},
    {"id": "P4", "source": "Shipping", "text": "Lost shipments may be replaced after 7 business days. Refunds for lost shipments require a carrier trace."},
]


def tokens(text):
    # Turn a sentence into lowercase words. Drop punctuation and tiny words.
    cleaned = [word.strip(".,!?;:"'").lower() for word in text.split()]
    return [word for word in cleaned if len(word) > 2]


def retrieve(query, k=2):
    """Retrieve: score chunks by shared words. Crude on purpose so a near miss is visible."""
    query_terms = set(tokens(query))
    scored = []
    for chunk in CHUNKS:
        overlap = query_terms & set(tokens(chunk["text"]))
        scored.append((len(overlap), chunk))
    scored.sort(key=lambda item: item[0], reverse=True)
    return scored[:k]  # top-k pages for this ticket


def answer(query):
    # Augment + generate: answer only from the top chunk. Score 0 → refuse/escalate.
    hits = retrieve(query, k=1)
    score, chunk = hits[0]
    if score == 0:
        return {"answer": "I cannot ground this. Escalate.", "citation": None, "score": 0}
    return {
        "answer": chunk["text"],
        "citation": f"{chunk['source']} ({chunk['id']})",
        "score": score,
    }


print("Four chunks loaded:", [chunk["id"] for chunk in CHUNKS])
print("STOP. Do not run the next cell until we guess.")


## This cell — the four pages we can cite. Look, then STOP.

| Id | Source | What it allows |
| --- | --- | --- |
| P1 | Refund | Damaged item, 30 days, cash of the order total |
| P2 | Refund | Change of mind: **no cash**. Store credit in 14 days |
| P3 | Privacy | Do not paste addresses or payment numbers into a public chatbot |
| P4 | Shipping | Lost shipment: replace after 7 business days; refund needs a carrier trace |

What the last cell built (RAG inside **observe**):

1. `CHUNKS` — **split + index**. Four pages. Each has an id so we can cite it.
2. `tokens` — lowercase, drop punctuation and tiny words, so "Refund!" matches "refund".
3. `retrieve` — count **shared words**, keep top-*k*. Crude on purpose. A DAT 5565 embedding can replace this later; still a tool.
4. `answer` — cite the top page, or if score is 0 **escalate**. The model may not invent.

We will not add a "+0.1 safety boost" on the words "not allowed." Today's point is the opposite: a wrong citation is a **management** problem, not a scoring trick.

STOP. Next fill: a question that is **not** on those four pages.


## Fill 1 — STOP. Do not scroll.

A vendor asks: **"What is our SOC 2 status?"** Nothing in P1–P4 mentions SOC 2.

Guess to a neighbor (20 seconds): will the agent **invent** an answer, **refuse**, or **escalate**? Two voices share. Then type. Then we run the three questions.


In [ ]:
# FILL WITH THE CLASS. >>> YOU FILL THIS <<<
# Fill 1 of 5. Type ONE word between the quotes, then run this cell.
# Allowed words: invent    refuse    escalate
# The vendor asked about SOC 2. Nothing in P1–P4 mentions SOC 2.
guess_soc2 = ""  # YOU TYPE HERE  (invent, refuse, or escalate)
print("You said:", guess_soc2 or "(say it, then type it)")
print("Next cell runs Maya, Jordan, and the SOC 2 question. Watch the citation.")


## This cell — three questions. RUN WITH ME


In [ ]:
# RUN WITH ME. Do not edit. Three tickets through retrieve → answer.
# Watch citation= and score=. A miss (score 0) is still a correct action a.
for query in [
    "Can we cash-refund a damaged mug after 10 days?",           # should cite P1
    "Customer wants cash back because they changed their mind.",  # should cite P2
    "What is our vendor's SOC 2 status?",                        # nothing matches
]:
    result = answer(query)
    print(f"Q: {query}")
    print(f"   score={result['score']} citation={result['citation']}")
    print(f"   {result['answer']}\n")


## This cell — why those scores. RUN WITH ME

Shared words, not magic. Watch the SOC 2 row: cleaned words, then a dash on every page.


In [ ]:
# RUN WITH ME. Do not edit. Why those scores?
for query in [
    "Can we cash-refund a damaged mug after 10 days?",
    "Customer wants cash back because they changed their mind.",
    "What is our vendor's SOC 2 status?",
]:
    q = set(tokens(query))
    print("Q:", query)
    print("   cleaned:", sorted(q))
    for chunk in CHUNKS:
        shared = q & set(tokens(chunk["text"]))
        print(f"   {chunk['id']} score={len(shared)}  shared={sorted(shared) or '—'}")
    print()
print("SOC 2 shares nothing. Score 0 is the right observe. Action a is refuse or escalate.")


**Family check.** Say this out loud: this is the **LLM** family. An RL-based agent cannot cite P1. A DAT 5565 embedding can rank chunks — that is a **tool**, not the agent. The agent still has to act (refund, refuse, escalate) and take feedback: was the citation real?

STOP. Next: short-term memory. Then the near miss.


## Short-term memory — still the agent. RUN WITH ME

The policy pages are **long-term memory** (the corpus). After retrieve, the agent must **hold** the citation while it plans the next step. If it drops the citation, the next step invents a window — same as Monday's chatbot inventing $999.

![Hold the citation](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/desk-memory.png)

A Q-table is not this job. Run this cell. Do not edit it.


In [ ]:
# RUN WITH ME. Do not edit.
# Long-term memory = CHUNKS (the binder). Short-term = working_memory (the sticky).
working_memory = {"citation": None, "plan": ["retrieve", "cite or miss", "recommend", "release"]}

print("Plan (decide):", " → ".join(working_memory["plan"]))
for query in [
    "Can we cash-refund a damaged mug after 10 days?",  # sticky should hold P1
    "What is our vendor's SOC 2 status?",               # sticky stays blank → escalate
]:
    found = answer(query)
    working_memory["citation"] = found["citation"]  # hold the citation while we plan
    next_step = "recommend, then release" if found["citation"] else "escalate (empty memory)"
    print(f"Q: {query}")
    print(f"   hold: {working_memory['citation']}")
    print(f"   next: {next_step}\n")
print("STOP. If the sticky is blank, the agent must not send cash.")


**Family check.** Memory here is the citation the LLM-based agent carries. It is not a Q-table. STOP. Do not scroll to the near miss until I say.


## Fill 2 — STOP. A wrong chunk is a management problem.

![Near miss on the word refund](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/desk-near-miss.png)

Question for the retriever: **"Is a refund allowed?"** Both P1 (damage) and P2 (change of mind) share the word refund.

Guess to a neighbor: which chunk comes **first**, P1 or P2? Two voices. Then type. If you speak the wrong sentence to a customer, you have unapproved policy — same cash leak as Monday's $999 chatbot.


In [ ]:
# FILL WITH THE CLASS. >>> YOU FILL THIS <<<
# Fill 2 of 5. Type P1 or P2 between the quotes, then run.
# Question: "Is a refund allowed?" Both chunks share the word refund.
guess_near = ""  # YOU TYPE HERE  (P1 or P2)
print("You said first chunk:", guess_near or "(P1 or P2, then we run it)")


## This cell — near miss. RUN WITH ME


In [ ]:
# RUN WITH ME. Do not edit. Print the top TWO chunks (k=2) so a near miss is visible.
print("Near miss for: Is a refund allowed?")
for score, chunk in retrieve("Is a refund allowed?", k=2):
    print(f"  score={score} {chunk['id']}: {chunk['text']}")
print("A citation is not enough. Ask: did that sentence answer THIS question?")


Managers should ask:

- Did the chunk come from the system of record, or from an old PDF?
- Does the reply cite the chunk id?
- What is the cost of the wrong chunk (cash out the door, privacy, a bad promise)?

We will **not** patch this with a +0.1 boost on "not allowed." That hides the problem. The question was too vague, so two opposite rules tied. The reviewer (next fill) owns whether we speak either sentence.

**Family check.** A Q-table cannot veto a bad citation. Hybrid later: learned policy on the usual case; language agent plus a reviewer on the document.

STOP. Next fill: who owns release?


## Fill 3 — STOP. Who releases the action?

Priya, order 1003: "Should we refund, and who signs?"

| Role | Owns | Must pass on |
| --- | --- | --- |
| Researcher | The cited chunk | citation id or an explicit miss |
| Analyst | The recommendation | action plus the chunk it used |
| Reviewer | The release decision | approve only if the citation is real |

Guess to a neighbor: who **releases** the refund — researcher, analyst, or reviewer? Two voices. Then type.


In [ ]:
# FILL WITH THE CLASS. >>> YOU FILL THIS <<<
# Fill 3 of 5. Type ONE role between the quotes, then run.
# Allowed words: researcher    analyst    reviewer
# Who is allowed to send action a (the refund stamp)?
who_releases = ""  # YOU TYPE HERE  (researcher, analyst, or reviewer)
print("Release belongs to:", who_releases or "(say it, then type it)")


## This cell — three-role handoff. RUN WITH ME


In [ ]:
# RUN WITH ME. Do not edit. These are steps in decide, not three new hires.
def researcher(query):
    # Observe: retrieve a chunk (or an explicit miss).
    result = answer(query)
    return {"role": "researcher", "citation": result["citation"], "evidence": result["answer"]}


def analyst(research):
    # Decide a recommended action a, but do not stamp it yet.
    if research["citation"] is None:
        action = "escalate"
    elif "30 days" in research["evidence"]:
        action = "apply the 30-day damage rule; do not invent a new window"
    else:
        action = "follow the cited sentence only"
    return {"role": "analyst", "action": action, "used": research["citation"]}


def reviewer(research, analysis):
    # Release: approve only if the citation is a real chunk in CHUNKS.
    known_ids = {chunk["id"] for chunk in CHUNKS}
    cited = research["citation"] or ""
    has_real_source = any(chunk_id in cited for chunk_id in known_ids)
    approved = has_real_source and analysis["used"] == research["citation"]
    return {"role": "reviewer", "approved": approved, "reason": "citation matches the corpus" if approved else "missing citation"}


query = "Damaged item refund window"
research = researcher(query)
analysis = analyst(research)
review = reviewer(research, analysis)
print(research)
print(analysis)
print(review)

# Broken handoff: the analyst cites nothing. The reviewer must stop the release.
bad_analysis = {"role": "analyst", "action": "refund everything", "used": None}
print("Broken handoff:", reviewer(research, bad_analysis))


**Family check.** Same as Monday's unknown id 9999: a person (or a reviewer role) owns the exception. The researcher does not get to send cash.

STOP. Next fill is **your** question.


## Fill 4 — STOP. Your question, in your own words.

Guess to a neighbor what you will ask. Two voices. Then type. If the citation is wrong or missing, say so out loud. That is the management problem.


In [ ]:
# FILL WITH THE CLASS. >>> YOU FILL THIS <<<
# Fill 4 of 5. Type a customer question between the quotes, then run.
# Example shape: "What is the damaged-item refund window?"
# After it runs: look at the shared-word line. Then say yes or no: did THAT page answer YOU?
my_question = ""  # YOU TYPE HERE
if not my_question.strip():
    print("Type a question, then run this cell again.")
else:
    q = set(tokens(my_question))
    print("Cleaned words:", sorted(q))
    for chunk in CHUNKS:
        shared = q & set(tokens(chunk["text"]))
        print(f"  {chunk['id']} score={len(shared)}  shared={sorted(shared) or '—'}")
    found = answer(my_question)
    print()
    print("Citation:", found["citation"])
    print(found["answer"])
    print("Did the sentence actually answer your question? Say yes or no out loud.")


## Practice — one business problem, real CFPB tickets. RUN WITH ME

**Business problem.** You sit on a **credit-card operations desk**. The **environment** is a public queue of complaints from the [CFPB Consumer Complaint Database](https://www.consumerfinance.gov/data-research/consumer-complaints/) (CC0). Each row is a state *s*. The **agent** observes with RAG against public US policy (not the model's memory), decides, sends action *a*, and takes reward *r*: was the citation real, and did we leak cash? The next complaint is *s′*.

Monday was Lumen Home. Same loop. Real tickets. Do not paste a customer's SSN or an employer file into a public model.

![Walk one ticket: agent vs environment](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/desk-agent-env.png)

We will walk **complaint 9999983** (JPMorgan Chase, card opened without consent) box by box. Then we run the rest of the queue.


In [ ]:
# RUN WITH ME. Do not edit. Load the environment (the CFPB queue).
# Agent = retrieve + decide. Environment = this CSV + leakage + the next ticket.
from pathlib import Path
import csv
import urllib.request


def course_data(name):
    """Local folder, Canvas data/, or GitHub — so Colab still runs."""
    for base in (Path("data"), Path("../../assets/data"), Path("assets/data"), Path("/content/data")):
        path = base / name
        if path.exists():
            return path
    url = (
        "https://raw.githubusercontent.com/salihtutun/"
        "Agentic-AI_Autonomous_Decision_Making/main/assets/data/" + name
    )
    dest_dir = Path("/content") if Path("/content").exists() else Path(".")
    dest = dest_dir / name
    urllib.request.urlretrieve(url, dest)
    return dest


# Public US rules the ops desk may cite. RAG lives here, inside the agent.
CHUNKS = [
    {"id": "FCBA", "source": "Fair Credit Billing Act", "text": "A cardholder may dispute a billing error in writing within 60 days of the statement. The issuer must investigate. Do not treat this as a change-of-mind refund."},
    {"id": "FCRA", "source": "Fair Credit Reporting Act", "text": "A consumer may dispute incorrect information on a credit report. The furnisher and the bureau must investigate. Do not silently delete a tradeline to make the ticket go away."},
    {"id": "FRAUD", "source": "Identity theft / card opened without consent", "text": "If a card was opened without the consumer's consent, freeze new charges, file an identity-theft report, and escalate to a person. Do not auto-refund from a chatbot."},
    {"id": "PRIV", "source": "Privacy", "text": "Do not paste Social Security numbers, full card numbers, or employer case files into a public chatbot."},
]

tickets = list(csv.DictReader(course_data("cfpb-credit-card-tickets.csv").open(encoding="utf-8")))
print("ENVIRONMENT  loaded", len(tickets), "real CFPB credit-card tickets.")
print("columns:", ", ".join(tickets[0].keys()))
print()
print("The queue (each row will be a state s):")
for row in tickets:
    print(f"  {row['complaint_id']:>8}  {row['company'][:28]:28}  {row['issue'][:40]}")
print("STOP. Next cell walks Chase 9999983 through s, observe, decide, a, r, s'.")


## This cell — Case 1, Chase 9999983, one full loop. RUN WITH ME

Say the labels out loud as they print: **environment**, **agent**, **state s**, **action a**, **reward r**, **next state s′**.


In [ ]:
# RUN WITH ME. One real ticket, one loop. Do not edit.
def query_for(row):
    """State the agent can see: narrative plus the CFPB issue labels."""
    return " ".join(
        part for part in (row["ticket_text"], row["issue"], row.get("sub_issue") or "") if part
    )


def decide_cfpb(found):
    """Policy: cite the rule, or escalate. Fraud is never an auto-refund."""
    cited = found["citation"] or ""
    if found["citation"] is None:
        return "escalate", "honest miss — nothing in the corpus"
    if "FRAUD" in cited:
        return "escalate to a person; freeze charges", "fraud is not a chatbot refund"
    if "FCRA" in cited:
        return "open a credit-report dispute; do not silently delete", "follow FCRA"
    if "FCBA" in cited:
        return "open a billing-error investigation", "follow FCBA"
    return "escalate", "citation is real but is not a ticket rule"


def answers_this_state(row, found):
    """A real chunk can still be the wrong page for this ticket (near miss)."""
    issue = f"{row['issue']} {row.get('sub_issue') or ''} {row['ticket_text']}".lower()
    cited = found["citation"] or ""
    if found["citation"] is None:
        return False
    if "FRAUD" in cited:
        return "consent" in issue
    if "FCRA" in cited:
        return "report" in issue
    if "FCBA" in cited:
        return "billing" in issue
    return False


def reward_cfpb(row, found, action):
    """r = +1 only if the citation is real AND it answers this state. Leakage is -1."""
    leaked = ("refund" in action.lower()) and ("FRAUD" in (found["citation"] or ""))
    if leaked:
        return -1, "leakage: chatbot refunded a fraud ticket"
    if found["citation"] is None:
        return 0, "honest miss / escalate — still a correct action a"
    if not answers_this_state(row, found):
        return 0, "citation is in the corpus, but it does not answer THIS state"
    return 1, "citation is real and it answers this ticket; leakage $0"


def walk_loop(row, next_row):
    """Print every box: environment -> agent -> environment."""
    print("=" * 60)
    print("ENVIRONMENT  gives state s")
    print(f"  complaint_id {row['complaint_id']}   {row['company']}")
    print(f"  issue: {row['issue']}")
    if row.get("sub_issue"):
        print(f"  sub_issue: {row['sub_issue']}")
    print(f"  ticket: {row['ticket_text']}")
    print()
    found = answer(query_for(row))
    print("AGENT  observes (RAG: retrieve against public US policy)")
    print(f"  citation={found['citation']}  score={found['score']}")
    print(f"  held in memory: {found['answer']}")
    print(f"  answers THIS state? {answers_this_state(row, found)}")
    print()
    action, why = decide_cfpb(found)
    print("AGENT  decides, then sends action a into the environment")
    print(f"  a = {action}")
    print(f"  why: {why}")
    print()
    r, note = reward_cfpb(row, found, action)
    print("ENVIRONMENT  returns reward r")
    print(f"  r = {r}   ({note})")
    print()
    print("ENVIRONMENT  gives next state s'")
    if next_row is None:
        print("  queue empty")
    else:
        print(f"  {next_row['complaint_id']}  {next_row['company']}  {next_row['issue']}")
    print("=" * 60)
    return {"s": row["complaint_id"], "a": action, "r": r, "s_prime": None if next_row is None else next_row["complaint_id"]}


case = next(row for row in tickets if row["complaint_id"] == "9999983")
case_index = tickets.index(case)
nxt = tickets[case_index + 1] if case_index + 1 < len(tickets) else None
walk_loop(case, nxt)
print("Family check: this ticket is a document. LLM family. A bandit cannot cite FRAUD.")


## This cell — the rest of the queue, same loop. RUN WITH ME

Every remaining row is a new *s*. Watch for a ticket whose retrieved chunk does **not** answer that complaint — same management problem as the near miss on "refund."


In [ ]:
# RUN WITH ME. Same agent, same environment, next tickets.
print(f"{'s (id)':>10}  {'company':<22}  {'citation':<22}  {'fits':<5}  {'r':>2}  s'")
print("-" * 88)
for i, row in enumerate(tickets):
    found = answer(query_for(row))
    action, _why = decide_cfpb(found)
    r, _note = reward_cfpb(row, found, action)
    fits = "yes" if answers_this_state(row, found) else "no"
    s_prime = tickets[i + 1]["complaint_id"] if i + 1 < len(tickets) else "—"
    cite = (found["citation"] or "miss / escalate")[:22]
    print(f"{row['complaint_id']:>10}  {row['company'][:22]:<22}  {cite:<22}  {fits:<5}  {r:>2}  {s_prime}")
print()
print("STOP. Point at one row: who is the agent, what is s, what is a, what is r, what is s'.")
print("Capital One advertising may retrieve a chunk that does not answer the ticket. Say so out loud.")


**Family check.** A credit-card complaint is a **document**. The CSV is the **environment**. RAG is a tool *inside the agent*. That is the LLM family. An RL agent cannot cite FCBA. Session 4's repeated FX choice is the other family.

## Fill 5 — STOP. Add this to your framework.

![Documents versus a repeated number](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/desk-two-families.png)

Guess to a neighbor: after today, which family owns **documents** in your project — LLM, RL, or hybrid? Two voices. Then type one sentence you will keep.

Hint: documents and exceptions go through retrieval plus a reviewer. A repeated number (overnight FX sleeve, daily order quantity) waits for Session 4. A DAT 5565 model may rank chunks; it is a tool, not the agent.


In [ ]:
# FILL WITH THE CLASS. >>> YOU FILL THIS <<<
# Fill 5 of 5. Two lines. Type, then run.
# Line 1: LLM, RL, or hybrid  (documents today are which family?)
# Line 2: one sentence you will keep on your project page
framework_family = ""  # YOU TYPE HERE  (LLM, RL, or hybrid)
framework_line = ""    # YOU TYPE HERE  (one sentence)
print("Family:", framework_family or "(say it, then type it)")
print("I will keep:", framework_line or "(one sentence)")


## Walking-out answers — we say these only after the five fills

1. If nothing matches (SOC 2), the **agent** refuses or escalates. That is still action *a*. Do not invent.
2. Overlap on "refund" can rank P1 first. A citation in memory is not enough — did it answer **this** state?
3. The **reviewer** owns release of *a*. The plan is decide, not a new org chart.
4. Retrieve, memory, and plan are how this LLM-based agent observes and decides. The CFPB CSV is the environment, not the agent. Session 4's bandit is the RL family. Both are Agentic AI. Do not mix them because both are "AI."
5. Your framework line: documents and exceptions → this LLM-based agent (retrieve + memory + reviewer). Repeated numeric choice → RL. Hybrid later = learned policy on the usual case, language agent plus a person on the exception.

Fits: high-volume policy lookup with a system of record. Does not fit: a negotiated exception a manager owns, or a repeated inventory quantity.
